In [17]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from transformers import (
    AutoTokenizer, 
    AutoImageProcessor, 
    ViTModel, 
    Wav2Vec2Model, 
    AutoFeatureExtractor
)
from datasets import load_dataset
from accelerate import Accelerator
import math

#  MULTI-STREAM INGESTOR 
class GeminiDataStreamer(IterableDataset):
    def __init__(self, mode="sft"):
        self.mode = mode
        self.tokenizer = AutoTokenizer.from_pretrained("gpt2")
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.v_proc = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224", use_fast=True)
        self.a_proc = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")
        
        # Primary Multimodal Streams
        self.v_stream = load_dataset("HuggingFaceH4/llava-instruct-mix-vsft", split="train", streaming=True)
        self.a_stream = load_dataset("PolyAI/minds14", "en-US", split="train", streaming=True)
        
        if mode == "dpo":
            # Using Argilla's DPO pairs 
            self.dpo_stream = load_dataset("argilla/distilabel-intel-orca-dpo-pairs", split="train", streaming=True)

    def _extract_text(self, item):
        if isinstance(item, str): return item
        if isinstance(item, list) and len(item) > 0:
            if isinstance(item[0], dict) and 'text' in item[0]:
                return " ".join([i['text'] for i in item if 'text' in i])
        return str(item)

    def __iter__(self):
        v_iter = iter(self.v_stream)
        a_iter = iter(self.a_stream)
        d_iter = iter(self.dpo_stream) if self.mode == "dpo" else None
        
        while True:
            try:
                v_item = next(v_iter)
                a_item = next(a_iter)
                
                # Image Prep
                img = v_item.get('image') or (v_item.get('images')[0] if v_item.get('images') else None)
                if img is None: continue
                pixels = self.v_proc(images=img.convert("RGB"), return_tensors="pt")['pixel_values'].squeeze(0)
                
                # Audio Prep
                audio = self.a_proc(a_item['audio']['array'], sampling_rate=16000, return_tensors="pt")['input_values'].squeeze(0)

                if self.mode == "sft":
                    # Extract SFT Text
                    raw_text = ""
                    if 'messages' in v_item:
                        raw_text = next((m['content'] for m in reversed(v_item['messages']) if m['role'] == 'assistant'), "")
                    elif 'conversations' in v_item:
                        raw_text = v_item['conversations'][-1]['value']
                    
                    text_str = self._extract_text(raw_text)
                    if not text_str: continue
                    
                    tokens = self.tokenizer(text_str, max_length=128, truncation=True, padding="max_length", return_tensors="pt")['input_ids'].squeeze(0)
                    yield pixels, tokens, audio
                
                else: # DPO MODE
                    d_item = next(d_iter)
                    # Chosen/Rejected from Orca DPO
                    c_txt = d_item['chosen']
                    r_txt = d_item['rejected']
                    
                    c_tokens = self.tokenizer(c_txt, max_length=128, truncation=True, padding="max_length", return_tensors="pt")['input_ids'].squeeze(0)
                    r_tokens = self.tokenizer(r_txt, max_length=128, truncation=True, padding="max_length", return_tensors="pt")['input_ids'].squeeze(0)
                    
                    yield pixels, c_tokens, r_tokens, audio

            except StopIteration: break
            except Exception: continue

# --- 2.  ARCHITECTURE ---
class SparseMoE(nn.Module):
    def __init__(self, d_model, num_experts=4):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model*2), nn.SiLU(), nn.Linear(d_model*2, d_model)) 
            for _ in range(num_experts)
        ])

    def forward(self, x):
        b, s, d = x.shape
        flat_x = x.view(-1, d)
        probs = F.softmax(self.gate(flat_x), dim=-1)
        weights, idx = torch.topk(probs, 1, dim=-1)
        
        out = torch.zeros_like(flat_x)
        for i, expert in enumerate(self.experts):
            mask = (idx == i).squeeze()
            if mask.any():
                out[mask] = expert(flat_x[mask]) * weights[mask]
        return out.view(b, s, d), probs.mean(0).std()

class GeminiLite(nn.Module):
    def __init__(self, vocab_size, d_model=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224")
        self.audio_enc = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        self.v_proj = nn.Linear(768, d_model)
        self.a_proj = nn.Linear(768, d_model)
        self.moe = SparseMoE(d_model)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, tokens, pixels, audio):
        t_feat = self.token_emb(tokens)
        with torch.no_grad():
            v_feat = self.v_proj(self.vit(pixels).last_hidden_state)
            a_feat = self.a_proj(self.audio_enc(audio).last_hidden_state)
        
        x = torch.cat([a_feat, v_feat, t_feat], dim=1)
        x_moe, aux_loss = self.moe(self.norm(x))
        return self.head(x + x_moe), aux_loss

# 3. EXECUTION 
def train():
    acc = Accelerator()
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    model = GeminiLite(len(tokenizer))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
    
    # SFT
    print(">>> Phase 1: SFT...")
    sft_loader = DataLoader(GeminiDataStreamer(mode="sft"), batch_size=1)
    model, opt, sft_loader = acc.prepare(model, opt, sft_loader)
    
    model.train()
    for i, (p, t, a) in enumerate(sft_loader):
        opt.zero_grad()
        logits, aux = model(t, p, a)
        loss = F.cross_entropy(logits[:, -128:-1, :].reshape(-1, len(tokenizer)), t[:, 1:].reshape(-1))
        acc.backward(loss + 0.01 * aux)
        opt.step()
        if i % 2 == 0: print(f"SFT {i} | Loss: {loss.item():.4f}")
        if i >= 400: break

    # DPO
    print("\n>>> Phase 2: DPO Alignment...")
    dpo_loader = DataLoader(GeminiDataStreamer(mode="dpo"), batch_size=1)
    dpo_loader = acc.prepare(dpo_loader)
    
    for i, (p, c, r, a) in enumerate(dpo_loader):
        opt.zero_grad()
        l_c, _ = model(c, p, a)
        l_r, _ = model(r, p, a)
        
        log_p_c = F.log_softmax(l_c, -1).mean()
        log_p_r = F.log_softmax(l_r, -1).mean()
        # DPO Loss Beta = 0.1
        dpo_loss = -F.logsigmoid(0.1 * (log_p_c - log_p_r))
        
        acc.backward(dpo_loss)
        opt.step()
        if i % 2 == 0: print(f"DPO {i} | Loss: {dpo_loss.item():.4f}")
        if i >= 200: break

    print(">>> Pipeline Finished Successfully.")
    print(">>> Pipeline Finished Successfully.")

    
    trainable_weights = {
        "v_proj": model.v_proj.state_dict(),
        "a_proj": model.a_proj.state_dict(),
        "moe": model.moe.state_dict(),
        "head": model.head.state_dict()
    }
    torch.save(trainable_weights, "gemini_adapters.pth")
    print("Multimodal adapters saved to gemini_adapters.pth")
if __name__ == "__main__":
    train()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


>>> Phase 1: SFT...


...

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

SFT 0 | Loss: 10.5390
SFT 2 | Loss: 11.0116
SFT 4 | Loss: 10.4445
SFT 6 | Loss: 10.4012
SFT 8 | Loss: 10.3607
SFT 10 | Loss: 11.3684
SFT 12 | Loss: 10.7483
SFT 14 | Loss: 10.2501
SFT 16 | Loss: 11.1298
SFT 18 | Loss: 10.1641
SFT 20 | Loss: 11.0384
SFT 22 | Loss: 10.0733
SFT 24 | Loss: 10.9853
SFT 26 | Loss: 9.9947
SFT 28 | Loss: 9.9443
SFT 30 | Loss: 10.9325
SFT 32 | Loss: 9.8436
SFT 34 | Loss: 9.7923
SFT 36 | Loss: 9.7348
SFT 38 | Loss: 9.6796
SFT 40 | Loss: 9.6182
SFT 42 | Loss: 9.5579
SFT 44 | Loss: 10.7281
SFT 46 | Loss: 9.4251
SFT 48 | Loss: 10.9911
SFT 50 | Loss: 9.2916
SFT 52 | Loss: 9.2193
SFT 54 | Loss: 9.1447
SFT 56 | Loss: 9.0672
SFT 58 | Loss: 10.6506
SFT 60 | Loss: 8.9125
SFT 62 | Loss: 8.8286
SFT 64 | Loss: 10.9118
SFT 66 | Loss: 8.6516
SFT 68 | Loss: 8.5597
SFT 70 | Loss: 10.9647
SFT 72 | Loss: 8.3669
SFT 74 | Loss: 8.2699
SFT 76 | Loss: 8.2506
SFT 78 | Loss: 10.9364
SFT 80 | Loss: 7.9438
SFT 82 | Loss: 10.9265
SFT 84 | Loss: 7.7281
SFT 86 | Loss: 7.6122
SFT 88 | Loss: 7

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

DPO 0 | Loss: 0.7271
DPO 2 | Loss: 0.7104
DPO 4 | Loss: 0.6932
DPO 6 | Loss: 0.6931
DPO 8 | Loss: 0.7357
DPO 10 | Loss: 0.7360
DPO 12 | Loss: 0.6890
DPO 14 | Loss: 0.7503
DPO 16 | Loss: 0.6981
DPO 18 | Loss: 0.7036
DPO 20 | Loss: 0.6759
DPO 22 | Loss: 0.6931
DPO 24 | Loss: 0.7638
DPO 26 | Loss: 0.6955
DPO 28 | Loss: 0.6932
DPO 30 | Loss: 0.6932
DPO 32 | Loss: 0.6932
DPO 34 | Loss: 0.7015
DPO 36 | Loss: 0.7195
DPO 38 | Loss: 0.6931
DPO 40 | Loss: 0.6931
DPO 42 | Loss: 0.6954
DPO 44 | Loss: 0.6981
DPO 46 | Loss: 0.6931
DPO 48 | Loss: 0.6931
DPO 50 | Loss: 0.6931
DPO 52 | Loss: 0.6932
DPO 54 | Loss: 0.7269
DPO 56 | Loss: 0.7447
DPO 58 | Loss: 0.6931
DPO 60 | Loss: 0.6509
DPO 62 | Loss: 0.6931
DPO 64 | Loss: 0.6948
DPO 66 | Loss: 0.7229
DPO 68 | Loss: 0.6932
DPO 70 | Loss: 0.6921
DPO 72 | Loss: 0.7294
DPO 74 | Loss: 0.6932
DPO 76 | Loss: 0.6703
DPO 78 | Loss: 0.6931
DPO 80 | Loss: 0.6931
DPO 82 | Loss: 0.6871
DPO 84 | Loss: 0.7361
DPO 86 | Loss: 0.7175
DPO 88 | Loss: 0.7357
DPO 90 | Loss: 

In [21]:
import torch
import torch.nn.functional as F
from PIL import Image
import requests
from transformers import AutoTokenizer, AutoImageProcessor, ViTModel, Wav2Vec2Model, AutoFeatureExtractor

class SparseMoE(nn.Module):
    def __init__(self, d_model, num_experts=4):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model*2), nn.SiLU(), nn.Linear(d_model*2, d_model)) 
            for _ in range(num_experts)
        ])

    def forward(self, x):
        b, s, d = x.shape
        flat_x = x.view(-1, d)
        probs = F.softmax(self.gate(flat_x), dim=-1)
        weights, idx = torch.topk(probs, 1, dim=-1)
        
        out = torch.zeros_like(flat_x)
        for i, expert in enumerate(self.experts):
            mask = (idx == i).squeeze()
            if mask.any():
                out[mask] = expert(flat_x[mask]) * weights[mask]
        return out.view(b, s, d), probs.mean(0).std()

class GeminiLite(nn.Module):
    def __init__(self, vocab_size, d_model=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224")
        self.audio_enc = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        self.v_proj = nn.Linear(768, d_model)
        self.a_proj = nn.Linear(768, d_model)
        self.moe = SparseMoE(d_model)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, tokens, pixels, audio):
        t_feat = self.token_emb(tokens)
        with torch.no_grad():
            v_feat = self.v_proj(self.vit(pixels).last_hidden_state)
            a_feat = self.a_proj(self.audio_enc(audio).last_hidden_state)
        
        x = torch.cat([a_feat, v_feat, t_feat], dim=1)
        x_moe, aux_loss = self.moe(self.norm(x))
        return self.head(x + x_moe), aux_loss


# 2. INITIALIZE GLOBAL MODEL
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = GeminiLite(len(tokenizer))

# Load the adapter dictionary
checkpoint = torch.load("/kaggle/working/gemini_adapters.pth")

# Map the saved weights to the model's internal modules
model.v_proj.load_state_dict(checkpoint['v_proj'])
model.a_proj.load_state_dict(checkpoint['a_proj'])
model.moe.load_state_dict(checkpoint['moe'])
model.head.load_state_dict(checkpoint['head'])

print("Successfully loaded multimodal adapters!")
model.eval()
# 3.GENERATION FUNCTION
@torch.no_grad()
def generate_response(model, question, image_path_or_url, max_new_tokens=20):
    v_proc = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
    a_proc = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")

    # Load Image
    if image_path_or_url.startswith("http"):
        img = Image.open(requests.get(image_path_or_url, stream=True).raw).convert("RGB")
    else:
        img = Image.open(image_path_or_url).convert("RGB")
    
    pixels = v_proc(images=img, return_tensors="pt")['pixel_values']
    audio = a_proc(torch.zeros(16000), sampling_rate=16000, return_tensors="pt")['input_values']
    
    # Encode Question
    input_ids = tokenizer.encode(question, return_tensors="pt")
    generated = input_ids

    for _ in range(max_new_tokens):
        logits, _ = model(generated, pixels, audio)
        next_token_logits = logits[:, -1, :]
        
        # Apply a bit of temperature for variety
        probs = F.softmax(next_token_logits / 0.8, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        generated = torch.cat([generated, next_token], dim=1)
        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(generated[0], skip_special_tokens=True)

# 4. RUN INFERENCE
test_image = "/kaggle/input/datasets/anwarshamim/images-for-test/images.jpg"
test_query = "explain this"

print("Generating...")
answer = generate_response(model, test_query, test_image)
print("-" * 30)
print(f"GEMINI LITE SAYS: {answer}")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Successfully loaded multimodal adapters!
Generating...


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


------------------------------
GEMINI LITE SAYS: explain thisuked instrument 1992 depositiant KwfolkマMeta TendWOOD mammoth exhibits username negotiatingiegpelled********eddyphabet
